In [2]:
import pandas as pd

#Carrego o banco de dados
Caminho = r"C:\Projeto de Limpeza\01_Data\archive (1).zip"
df = pd.read_csv(Caminho)

# Ja que algumas colunas possui erro o Pandas lê como testo, o codigo abaixo fais com que estas colunas seja lido como numeros
df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce')
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')

# Converte a data para formato datetime
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

# A coluna Total Spent(total gasto) possui alguns valore nulos(NaN) para resolver isso multipicamos Quantity(Quantidade) x Price Per Unit(Preço por Unidade)
df['Total Spent'] = df['Total Spent'].fillna(df['Quantity'] * df['Price Per Unit'])

# Remove as colunas que começam com 'Unnamed'
colunas_para_remover = ['Unnamed: 0.1', 'Unnamed: 0']
df = df.drop(columns=colunas_para_remover)

# Isso unifica 'ERROR' e 'UNKNOWN' com os nulos que já existem
valores_lixo = ['ERROR', 'UNKNOWN', 'nan']
df = df.replace(valores_lixo, pd.NA)

# Padronização de Texto (Garante que 'coffee' e 'Coffee' sejam iguais)
colunas_texto = ['Item', 'Payment Method', 'Location']
for col in colunas_texto:
    df[col] = df[col].str.lower().str.strip()

# Recuperação de Dados Numéricos (Lógica Matemática)
# Se falta o preço, mas tem o total e a quantidade:
df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Total Spent'] / df['Quantity'])

# Se falta a quantidade, mas tem o total e o preço:
df['Quantity'] = df['Quantity'].fillna(df['Total Spent'] / df['Price Per Unit'])

# O que sobrar de texto vira "não informado"
df[colunas_texto] = df[colunas_texto].fillna('não informado')

# Problema: 'credit card' vs 'não informado'
df['Payment Method'] = df['Payment Method'].str.lower().str.strip()
df['Payment Method'] = df['Payment Method'].replace({
    'credit card': 'Cartão de Crédito',
    'cash': 'Dinheiro',
    'digital wallet': 'Carteira Digital',
    'não informado': 'Não Informado',
    'credit': 'Cartão de Crédito', 
    'card': 'Cartão de Crédito'
})

# Problema: 'coffee', 'cake' (inglês) misturado com 'não informado'
df['Item'] = df['Item'].str.lower().str.strip()
df['Item'] = df['Item'].replace({
    'coffee': 'Café',
    'cake': 'Bolo',
    'cookie': 'Biscoito',
    'salad': 'Salada',
    'smoothie': 'Vitamina',
    'sandwich': 'Sanduíche',
    'juice': 'Suco',
    'tea': 'Chá',
    'não informado': 'Não Informado'
})

# Padronizar Location
df['Location'] = df['Location'].str.lower().str.strip()
df['Location'] = df['Location'].replace({
    'takeaway': 'Para Viagem',
    'in-store': 'Na Loja',
    'in store': 'Na Loja',  # caso exista sem hífen
    'não informado': 'Não Informado',
    'nao informado': 'Não Informado'  # variação sem acento
})

# Mantém os dados, mas documenta a limitação
print(f"ATENÇÃO: {df['Location'].value_counts()['Não Informado']} registros sem localização")

# Para coluna 'Total Spent'
Q1 = df['Total Spent'].quantile(0.25)  # 1º quartil (25%)
Q3 = df['Total Spent'].quantile(0.75)  # 3º quartil (75%)
IQR = Q3 - Q1  # Distância interquartil

# Definindo limites
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

# Identificando outliers
outliers = df[(df['Total Spent'] < limite_inferior) | (df['Total Spent'] > limite_superior)]
print(f"Outliers encontrados: {len(outliers)}")
print(outliers[['Total Spent', 'Item', 'Quantity']])

df_sem_outliers = df[df['Total Spent'] <= limite_superior]
print(f"Dataset original: {len(df)} registros")
print(f"Após remover outliers: {len(df_sem_outliers)} registros")

# Verifica quantas linhas são idênticas
print(f"Linhas duplicadas: {df.duplicated().sum()}")
# Se houver, remova:
df = df.drop_duplicates()

#Visualizo o banco de dados e identifico as colunas
print(df.dtypes)
print(df.head(100))
print("Colunas:", df.columns.tolist())

df.to_csv(r"C:\ESTUDOS_DE_DADOS\01_Data\vendas_cafeteria_limpo.csv", index=False)


ATENÇÃO: 3961 registros sem localização
Outliers encontrados: 268
      Total Spent           Item  Quantity
10           25.0         Salada       5.0
51           25.0         Salada       5.0
52           25.0  Não Informado       5.0
96           25.0         Salada       5.0
100          25.0  Não Informado       5.0
...           ...            ...       ...
9791         25.0         Salada       5.0
9805         25.0         Salada       5.0
9879         25.0         Salada       5.0
9908         25.0         Salada       5.0
9971         25.0         Salada       5.0

[268 rows x 3 columns]
Dataset original: 10000 registros
Após remover outliers: 9692 registros
Linhas duplicadas: 0
Transaction ID                 str
Item                           str
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
dtype: object
   Tran

## O Dataset Original demostrava os segintes problemas 

1. Erro de digitação/nomenclatura
2. Tipos de dados incorretos
3. Valores nulos/Nan (dados faltantes)
4. Inconsistência de idioma nas categorias
5. Formato inconsistente dentro da mesma categoria
6. 'Não informado' como valor misturado com categorias válidas
7. Erro na coluna Transaction Date
8. Possível inconsistência lógica

## O Que eu fis

1. tranformar algumas colunas em numeros e datatime ja que o Pandas lia tudo como texto
2. remover colunas indegejadas para o resultado final
3. calcular valores possiveis como quantidade X preço por unidades = total ou fazer os calculo inverso que seria dividir o total pela quantidade ou preço por unidade para encontrar o outo valor
4. transfomei ERROR e UNKNOWN em nulos(NaN)
5. Padronizei elementos como bolo e cake que são o mesmo item 
6. tirei linhas duplicatas